# Advanced RAG Systems — Three Variations on One Engine

Episode #1 built the production retrieval core: hybrid search + fusion + rerank ("Layer 1").
This notebook shows the three famous techniques that change ONE thing around that engine:

1. **HyDE** — changes the *input* (search with a hypothetical answer, not the question)
2. **GraphRAG (toy)** — changes the *index* (connected facts instead of a flat chunk list)
3. **Agentic RAG (toy loop)** — changes *how many times* the engine runs (retrieve → grade → retry)

Same 11-chunk Nimbus Cloud corpus, same embedding model as Episode #1.

In [ ]:
# Install the two libraries we need (run this cell first)
!pip install -q sentence-transformers rank-bm25

## Setup — the corpus and the embedding model

In [ ]:
# Numpy handles the vector math
import numpy as np
# re (regular expressions) - used later by the toy grader to look for a "%" figure
import re
# The embedding model: text in, 384-number meaning-vector out
from sentence_transformers import SentenceTransformer

# The same fictional Nimbus Cloud wiki as Episode #1: (chunk name, chunk text)
CORPUS = [
    ("sla_standard", "Nimbus Cloud Standard plan SLA: 99.9% monthly uptime guarantee. If monthly downtime exceeds 1 hour, customers receive a service credit of 10% of that month's bill. Credits are applied to the next invoice automatically."),
    ("sla_pro", "Nimbus Cloud Pro plan SLA: 99.95% monthly uptime. If monthly downtime exceeds 2 hours, customers receive a service credit of 15% of that month's bill. Pro customers also get priority support with a 4-hour response time."),
    ("sla_enterprise", "Nimbus Cloud Enterprise plan SLA: 99.99% monthly uptime guarantee. If monthly downtime exceeds 4 hours, enterprise customers receive a service credit of 25% of that month's bill. Enterprise includes a dedicated account manager and 24x7 phone support."),
    ("refund_policy", "Nimbus Cloud refund policy: subscription fees are non-refundable. However, if you cancel within the first 14 days of a new annual plan, you may request a full refund of the first month. SLA service credits are separate from refunds and do not constitute a refund."),
    ("enterprise_support", "Enterprise support at Nimbus Cloud includes 24x7 phone and email support, a dedicated account manager, quarterly architecture reviews, and a 15-minute response time for critical incidents. Enterprise support is included in the Enterprise plan at no extra cost."),
    ("downtime_definition", "How Nimbus Cloud measures downtime: downtime is counted when the status page shows a major outage affecting core APIs in your region. Scheduled maintenance windows announced 72 hours in advance are excluded from downtime calculations."),
    ("status_page", "The Nimbus Cloud status page shows real-time service health for all regions. Subscribe to status updates by email or webhook. Historical uptime for the last 90 days is published on the status page."),
    ("credits_process", "Service credits at Nimbus Cloud are calculated at the end of each billing month based on measured downtime. Credits appear automatically on the following invoice. Customers do not need to file a claim for SLA credits."),
    ("sla_overview", "Nimbus Cloud offers three plans with different SLA tiers. Higher tiers carry stronger uptime guarantees and larger service credits for extended outages. Full details of percentages and thresholds are listed on each plan's SLA page."),
    ("outage_postmortem", "Postmortem, March 2026 outage: Nimbus Cloud experienced 5.2 hours of downtime in the us-east region due to a database failover failure. Affected Enterprise customers received service credits under the Enterprise SLA. A full root-cause analysis was published to enterprise customers."),
    ("sla_faq", "SLA FAQ: What is an SLA? A service level agreement defines uptime guarantees. What happens if Nimbus misses the SLA? Customers receive service credits described in their plan's SLA. How do I check my plan? See the billing page. This FAQ summarizes SLA topics; exact percentages appear only on the plan SLA pages."),
]

# The question used across all three variations
QUERY = "What is the SLA refund percentage for enterprise downtime exceeding 4 hours?"

# Split the corpus into two parallel lists: names and texts
texts = [t for _, t in CORPUS]   # just the chunk texts
names = [n for n, _ in CORPUS]   # just the chunk names

# Load the small, free embedding model (downloads ~90 MB the first time)
emb = SentenceTransformer("all-MiniLM-L6-v2")

# Embed all 11 chunks once; normalize so dot product = cosine similarity
E = emb.encode(texts, normalize_embeddings=True)

# Helper: run dense (vector) search for any query text, return top-k (name, score)
def dense_top(query_text, k=3):
    # embed the query with the same model
    q = emb.encode([query_text], normalize_embeddings=True)[0]
    # one matrix multiply scores the query against all 11 chunks
    cos = E @ q
    # take the k highest scores, best first
    idx = list(np.argsort(-cos))[:k]
    # return (chunk name, score) pairs
    return [(names[i], float(cos[i])) for i in idx]

## Variation 1 — HyDE: change the input

A question and a document are different *shapes*, so their vectors match weakly.
HyDE (Hypothetical Document Embeddings) first asks an LLM to write a fake answer,
then searches with THAT text instead. The fake answer's facts may be wrong —
below it says 20%, the real answer is 25% — it is only a search key, never shown to the user.
(In this notebook the "LLM" is us, writing the fake answer by hand.)

In [ ]:
# The hypothetical answer an LLM might write (note: it invents 20% - the real figure is 25%)
HYDE_DOC = ("According to the Nimbus Cloud enterprise service level agreement, enterprise "
            "plan customers are eligible for a service credit of 20% of their monthly fee "
            "when total downtime in a billing month exceeds four hours. The credit is "
            "applied automatically to the next invoice after the SLA claim is verified.")

# Baseline: search with the raw QUESTION (this is Episode #1's naive failure)
print("searching with the QUESTION:")
for n, s in dense_top(QUERY):
    print(f"   {n:20s} {s:.4f}")

# HyDE: search with the HYPOTHETICAL ANSWER instead
print("searching with the HYPOTHETICAL ANSWER (wrong number inside!):")
for n, s in dense_top(HYDE_DOC):
    print(f"   {n:20s} {s:.4f}")

**Result:** the question ranked the answer-less FAQ #1 (0.6405). The hypothetical answer —
*even with the wrong percentage inside* — ranked the correct chunk #1 at 0.8689 and pulled all
three plan SLA chunks to the top. Document-shaped input matches documents better.

## Variation 2 — GraphRAG (toy): change the index

Some questions need TWO facts joined: *"What credit does the plan with 24x7 phone support
give for downtime over 4 hours?"* The plan's identity is in one chunk, its credit in another.
GraphRAG stores knowledge as connected facts — (subject, relation, object) triples —
and answers by *walking the connections*. Real GraphRAG extracts these triples from your
documents with an LLM; here they are written by hand to show the mechanism.

In [ ]:
# The knowledge graph as a list of (subject, relation, object) facts
GRAPH = [
    ("Enterprise plan", "has support", "24x7 phone and email support"),
    ("Pro plan", "has support", "business-hours email support"),
    ("Standard plan", "has support", "community forum support"),
    ("Enterprise plan", "credit when downtime > 4 hours", "25%"),
    ("Pro plan", "credit when downtime > 2 hours", "15%"),
    ("Standard plan", "credit when downtime > 1 hour", "10%"),
]

# Look up one fact in the graph
def graph_answer(plan, condition):
    # walk every triple
    for s, r, o in GRAPH:
        # match on the plan and the condition text
        if s == plan and condition in r:
            # found: return the object (the credit)
            return o
    return None

# The JOIN question needs two hops:
# hop 1 - which plan has 24x7 phone support?
plan = [s for s, r, o in GRAPH if r == "has support" and "24x7" in o][0]
print(f"step 1: which plan has 24x7 phone support? -> {plan}")
# hop 2 - that plan's credit for downtime > 4 hours
print(f"step 2: its credit for downtime > 4 hours? -> {graph_answer(plan, '> 4 hours')}")

# Compare: plain dense search on the same join question
print("dense search on that join question:")
for n, s in dense_top("What credit does the plan with 24x7 phone support give for downtime over 4 hours?"):
    print(f"   {n:20s} {s:.4f}")

**Result:** the graph answers in two clean hops: Enterprise plan → 25%.
Dense search on the same question doesn't surface the correct chunk in its top 3 at all —
no single chunk contains both facts, so no single vector can match the whole question.

## Variation 3 — Agentic RAG (toy loop): change how many times the engine runs

A fixed pipeline can't notice a bad retrieval. An agent loops:
**retrieve → grade → if bad, rewrite the query and retrieve again.**
In production the grader is an LLM judging relevance; here a simple rule stands in
("does the top chunk contain a % figure?") so the loop is visible without an API key.

In [ ]:
# The grader: PASS only if the TOP chunk contains an actual percentage figure
def grade(chunks):
    # find the top chunk's text and look for a pattern like "25%"
    return bool(re.search(r"\d+%", texts[names.index(chunks[0][0])]))

# Attempt 1: retrieve with the question as-is
top = dense_top(QUERY)
print(f"attempt 1: question as-is -> top-1 {top[0][0]}; grader (top chunk contains a %?): {grade(top)}")

# The grader failed -> the agent REWRITES the query and retries (max loop in real systems)
if not grade(top):
    # a rewritten, keyword-richer version of the question
    REWRITTEN = "enterprise SLA service credit percentage downtime exceeding 4 hours"
    top2 = dense_top(REWRITTEN)
    print(f"rewrite: '{REWRITTEN}'")
    print(f"attempt 2: -> top-1 {top2[0][0]} ({top2[0][1]:.4f}); grader: {grade(top2)}")

**Result:** attempt 1 lands on the FAQ and the grader rejects it — a pipeline would have
generated from it anyway. The agent rewrites, retrieves again, and the correct chunk wins.
The loop is the whole idea; the engine underneath didn't change at all.

## The takeaway

| Variation | What changed | Engine |
|---|---|---|
| HyDE | the input (question → fake answer) | same Layer 1 |
| GraphRAG | the index (flat chunks → connected facts) | same idea |
| Agentic | the control flow (one pass → loop with a grader) | same Layer 1 |

Fix the engine first — then change one thing around it at a time.